# SynthBA Brain Age - SIMON batch

**Before running:** Runtime -> Change runtime type -> **GPU (L4 or T4)**

This notebook runs `SynthBA` on the cleaned `SIMON` T1 manifest, saves progress to Google Drive after each scan, resumes safely by per-scan key, and can auto-disconnect the runtime when finished.

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone / pull repo
import os

REPO = '/content/faceage-to-brainage'

if not os.path.exists(REPO):
    os.system(f'git clone https://github.com/kondratevakate/faceage-to-brainage.git {REPO}')
else:
    os.system(f'git -C {REPO} pull --rebase')

os.chdir(REPO)
print('Working dir:', os.getcwd())

In [ ]:
# 3. Install runtime dependencies without a restart
import subprocess
import sys

for pkg in ['synthba', 'nibabel', 'pandas', 'tqdm']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    print(f'  {pkg}:', 'ok' if r.returncode == 0 else r.stderr[-200:])

print('Done')

In [ ]:
# 4. Locate SIMON manifest, normalize paths, and build a stable resume key
from pathlib import Path
import pandas as pd

DRIVE = '/content/drive/MyDrive'
DEVICE = 'cuda'
SYNC_EVERY = 1              # set to 5 if you want slightly less Drive I/O
AUTO_DISCONNECT_WHEN_DONE = True

SIMON_ROOT_CANDIDATES = [
    Path(DRIVE) / 'brain_mri' / 't1_only' / 'simon',
    Path(DRIVE) / 'brain_mri' / 'simon',
]

SIMON_ROOT = next((p for p in SIMON_ROOT_CANDIDATES if (p / 'manifest.csv').exists()), None)
if SIMON_ROOT is None:
    matches = [p.parent for p in Path(DRIVE).rglob('manifest.csv') if p.parent.name.lower() == 'simon']
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one simon/manifest.csv under {DRIVE}, found {len(matches)}: {matches[:5]}')
    SIMON_ROOT = matches[0]

MANIFEST_SRC = SIMON_ROOT / 'manifest.csv'
MANIFEST_COLAB = SIMON_ROOT / 'manifest_colab.csv'
IMAGES_DIR = SIMON_ROOT / 'images'
DRIVE_OUT = Path(DRIVE) / 'brain_mri' / 'brainage_results' / 'simon' / 'synthba_predictions.csv'

manifest = pd.read_csv(MANIFEST_SRC)
if 't1_path' not in manifest.columns and 't1_filename' not in manifest.columns:
    raise KeyError(f'Manifest must contain t1_path or t1_filename. Columns: {list(manifest.columns)}')

def resolve_t1_path(row):
    candidates = []
    if 't1_path' in row.index and pd.notna(row['t1_path']) and str(row['t1_path']).strip():
        raw = str(row['t1_path']).strip()
        candidates.extend([
            Path(raw),
            IMAGES_DIR / Path(raw).name,
            IMAGES_DIR / raw,
        ])
    if 't1_filename' in row.index and pd.notna(row['t1_filename']) and str(row['t1_filename']).strip():
        name = str(row['t1_filename']).strip()
        candidates.append(IMAGES_DIR / name)

    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    fallback_name = ''
    if candidates:
        fallback_name = Path(candidates[0]).name
    return str(IMAGES_DIR / fallback_name)

def clean_key_part(value):
    if value is None:
        return ''
    if isinstance(value, float) and pd.isna(value):
        return ''
    text = str(value).strip()
    return '' if text.lower() == 'nan' else text

def build_scan_key(row):
    parts = [
        clean_key_part(row.get('subject_id')),
        clean_key_part(row.get('session_id')),
        clean_key_part(row.get('run')),
        clean_key_part(row.get('acquisition_label')),
        Path(str(row['t1_path'])).name,
    ]
    return '|'.join([p for p in parts if p])

manifest['t1_path'] = manifest.apply(resolve_t1_path, axis=1)
missing_inputs = [p for p in manifest['t1_path'] if not Path(p).exists()]
if missing_inputs:
    raise FileNotFoundError(f'Missing {len(missing_inputs)} SIMON inputs. First 5: {missing_inputs[:5]}')

manifest['scan_key'] = manifest.apply(build_scan_key, axis=1)
manifest.to_csv(MANIFEST_COLAB, index=False)

print('SIMON_ROOT:', SIMON_ROOT)
print('Manifest rows:', len(manifest))
print('Manifest written:', MANIFEST_COLAB)
display_cols = [c for c in ['subject_id', 'session_id', 'run', 'acquisition_label', 't1_path', 'age', 'scan_key'] if c in manifest.columns]
print(manifest[display_cols].head(3).to_string())

In [ ]:
# 5. GPU sanity check
import subprocess
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

subprocess.run(['nvidia-smi'], check=False)

In [ ]:
# 6. Run SynthBA on SIMON with safe resume, per-scan checkpoints, and optional auto-disconnect
import gc
import math
import os
import shutil
import sys
from pathlib import Path

import nibabel as nib
import pandas as pd
import torch
from tqdm import tqdm

os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
sys.path.insert(0, REPO)

from src.brain_age import reset_synthba_import_state, suppress_synthba_import_warnings

reset_synthba_import_state()
suppress_synthba_import_warnings()

from synthba import SynthBA

OUT_CSV_PATH = Path('/tmp/simon_synthba_predictions.csv')
DRIVE_CSV_PATH = Path(DRIVE_OUT)
DRIVE_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
manifest = pd.read_csv(MANIFEST_COLAB)

def export_records(records_by_key):
    if not records_by_key:
        return pd.DataFrame()
    frame = pd.DataFrame(records_by_key.values())
    sort_cols = [c for c in ['session_id', 'run', 'acquisition_label', 'subject_id'] if c in frame.columns]
    if sort_cols:
        frame = frame.sort_values(sort_cols, kind='stable')
    frame.to_csv(OUT_CSV_PATH, index=False)
    shutil.copy2(OUT_CSV_PATH, DRIVE_CSV_PATH)
    return frame

records_by_key = {}
done = set()

resume_source = None
if DRIVE_CSV_PATH.exists():
    resume_source = DRIVE_CSV_PATH
elif OUT_CSV_PATH.exists():
    resume_source = OUT_CSV_PATH

if resume_source is not None:
    prev = pd.read_csv(resume_source)
    if 'scan_key' not in prev.columns:
        raise ValueError('Existing CSV is missing scan_key. Rename or delete it before resuming.')
    prev = prev.drop_duplicates(subset=['scan_key'], keep='last')
    records_by_key = {str(row['scan_key']): row.to_dict() for _, row in prev.iterrows()}
    done = set(prev.loc[prev['status'] == 'ok', 'scan_key'].astype(str))
    export_records(records_by_key)

print('already done:', len(done))

sba = SynthBA(DEVICE)
processed_since_sync = 0

try:
    for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc='SynthBA SIMON'):
        scan_key = str(row['scan_key'])
        if scan_key in done:
            continue

        chron_age = float(row['age']) if not pd.isna(row['age']) else float('nan')
        record = {
            'scan_key': scan_key,
            'subject_id': str(row.get('subject_id', '')),
            'session_id': row.get('session_id', ''),
            'run': row.get('run', ''),
            'acquisition_label': row.get('acquisition_label', ''),
            'chron_age': chron_age,
            'predicted_age': float('nan'),
            'brain_age_gap': float('nan'),
            'model_name': 'SynthBA',
            'status': 'error',
            'error': '',
            'input_path': str(row['t1_path']),
        }

        try:
            img = nib.load(str(row['t1_path']))
            pred = float(sba.run(img, preprocess=True, mr_weighting='t1'))
            record['predicted_age'] = pred
            if not math.isnan(chron_age):
                record['brain_age_gap'] = pred - chron_age
            record['status'] = 'ok'
            done.add(scan_key)
        except Exception as e:
            record['error'] = str(e)

        records_by_key[scan_key] = record
        processed_since_sync += 1

        if processed_since_sync >= SYNC_EVERY:
            saved = export_records(records_by_key)
            print(f'saved rows: {len(saved)}')
            processed_since_sync = 0

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    final_df = export_records(records_by_key)
    print('done')
    print('saved rows:', len(final_df))
finally:
    final_df = export_records(records_by_key)
    print('final backup:', DRIVE_CSV_PATH)
    print('final rows:', len(final_df))

    if AUTO_DISCONNECT_WHEN_DONE:
        try:
            from google.colab import runtime
            print('disconnecting runtime to stop GPU billing...')
            runtime.unassign()
        except Exception as e:
            print('could not auto-disconnect runtime:', e)

In [ ]:
# 7. Summary cell (run after reopening if auto-disconnect was enabled)
import numpy as np
import pandas as pd

df = pd.read_csv(DRIVE_OUT)
ok = df[df['status'] == 'ok'].copy()

print('rows total:', len(df))
print('rows ok:', len(ok))

if len(ok):
    err = ok['predicted_age'] - ok['chron_age']
    print('MAE:', float(err.abs().mean()))
    print('RMSE:', float(np.sqrt((err ** 2).mean())))
    print('Bias:', float(err.mean()))
    print('Predicted age std:', float(ok['predicted_age'].std()))
    print(ok[['subject_id', 'session_id', 'run', 'acquisition_label', 'chron_age', 'predicted_age', 'brain_age_gap', 'status']].tail(10).to_string())